# Raman Progress Tracker

This notebook scans Raman run directories, checks completion via `gecko.load_calc`, and tracks three stages:

- `madness_raman`
- `dalton_optimize`
- `dalton_raman`

In [ ]:
from __future__ import annotations

import contextlib
import io
import os
import re
import shlex
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAMAN_ROOT = Path('.').resolve()
DATA_ROOT = RAMAN_ROOT / 'data'
if not DATA_ROOT.exists():
    DATA_ROOT = RAMAN_ROOT

MRA_DIR = 'mra-p07'
OPTIMIZE_INPUT = 'optimize.dal'
RAMAN_INPUT = 'raman.dal'
OPT_MOL_TEMPLATE = 'opt_{mol}_{basis}.mol'
MADNESS_INPUT_OVERRIDE = None  # e.g. 'H2O_raman.in'

# Optional: set to your local gecko/src if gecko is not importable in this kernel.
GECKO_SRC_OVERRIDE = None  # e.g., Path('/path/to/gecko/src')

RAMAN_ROOT, DATA_ROOT

In [ ]:
def _candidate_gecko_src_paths(script_path: Path) -> list[Path]:
    out: list[Path] = []
    env_src = os.environ.get('GECKO_SRC')
    if env_src:
        out.append(Path(env_src).expanduser())
    if GECKO_SRC_OVERRIDE is not None:
        out.append(Path(GECKO_SRC_OVERRIDE).expanduser())

    for parent in script_path.resolve().parents:
        out.append(parent / 'gecko' / 'src')

    out.append(Path.cwd() / 'src')
    out.append(Path.cwd() / 'gecko' / 'src')

    unique: list[Path] = []
    seen: set[Path] = set()
    for p in out:
        try:
            rp = p.resolve()
        except Exception:
            continue
        if rp in seen:
            continue
        seen.add(rp)
        unique.append(rp)
    return unique


def bootstrap_load_calc() -> callable:
    try:
        from gecko.core.load import load_calc
        return load_calc
    except Exception:
        pass

    script_hint = RAMAN_ROOT / 'slurm_scripts' / 'submit_raman_jobs.py'
    for candidate in _candidate_gecko_src_paths(script_hint):
        if not candidate.exists():
            continue
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        try:
            from gecko.core.load import load_calc
            return load_calc
        except Exception:
            continue

    raise RuntimeError('Could not import gecko. Set GECKO_SRC or GECKO_SRC_OVERRIDE.')


def running_workdirs() -> set[Path]:
    user = os.environ.get('USER')
    if not user:
        return set()

    try:
        proc = subprocess.run(
            ['squeue', '-h', '-u', user, '-o', '%Z'],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        return set()

    if proc.returncode != 0:
        return set()

    out: set[Path] = set()
    for line in proc.stdout.splitlines():
        wd = line.strip()
        if wd:
            try:
                out.add(Path(wd).expanduser().resolve())
            except Exception:
                pass
    return out


def has_raman(calc) -> bool:
    data = getattr(calc, 'data', None)
    if not isinstance(data, dict):
        return False
    raman = data.get('raman')
    if raman is None:
        return False

    if isinstance(raman, dict):
        by_freq = raman.get('raman_by_freq')
        if isinstance(by_freq, dict) and by_freq:
            return True

        pol = raman.get('polarization_frequencies')
        if pol is not None:
            try:
                return len(pol) > 0
            except Exception:
                return True

        vib = raman.get('vibrational_frequencies')
        if vib is not None:
            try:
                return len(vib) > 0
            except Exception:
                return True

        return bool(raman)

    try:
        return len(raman) > 0
    except Exception:
        return True


def run_complete_raman(run_dir: Path, load_calc) -> tuple[bool, str, bool]:
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            calc = load_calc(run_dir)
    except Exception as exc:
        return False, f'gecko.load_calc failed: {type(exc).__name__}: {exc}', True

    if has_raman(calc):
        return True, 'raman present', False

    return False, 'calculation loaded but raman is missing', False


def find_madness_input(mra_dir: Path, molecule: str) -> Path | None:
    if not mra_dir.exists():
        return None

    if MADNESS_INPUT_OVERRIDE:
        p = mra_dir / MADNESS_INPUT_OVERRIDE
        if p.exists():
            return p
        return None

    preferred = mra_dir / f'{molecule}_raman.in'
    if preferred.exists():
        return preferred

    ins = sorted(mra_dir.glob('*.in'))
    if len(ins) == 1:
        return ins[0]
    if len(ins) > 1:
        for p in ins:
            if 'raman' in p.name.lower():
                return p
        return ins[0]

    return None


def select_base_mol_file(molecule: str, basis_dir: Path) -> Path | None:
    expected = basis_dir / f'{molecule}_{basis_dir.name}.mol'
    if expected.exists():
        return expected

    mols = sorted([p for p in basis_dir.glob('*.mol') if not p.name.startswith('opt_')])
    if len(mols) == 1:
        return mols[0]
    if len(mols) > 1:
        for p in mols:
            if p.stem.startswith(f'{molecule}_'):
                return p
        return mols[0]

    return None


def molecule_sort_key(name: str):
    m = re.fullmatch(r'n(\d+)', name)
    if m:
        return (0, int(m.group(1)))
    return (1, name)


def is_molecule_dir(path: Path) -> bool:
    if not path.is_dir():
        return False
    if (path / MRA_DIR).is_dir():
        return True

    for child in path.iterdir():
        if not child.is_dir() or child.name == MRA_DIR:
            continue
        if any(child.glob('*.mol')) and any(child.glob('*.dal')):
            return True
    return False

In [ ]:
load_calc = bootstrap_load_calc()
running = running_workdirs()

rows: list[dict] = []
mol_dirs = sorted([p for p in DATA_ROOT.iterdir() if is_molecule_dir(p)], key=lambda p: molecule_sort_key(p.name))

for mol_dir in mol_dirs:
    mol = mol_dir.name

    # MADNESS Raman stage
    mra_dir = mol_dir / MRA_DIR
    mad_in = find_madness_input(mra_dir, mol)

    if not mra_dir.exists() or mad_in is None:
        rows.append({
            'molecule': mol,
            'code': 'madness',
            'stage': 'raman',
            'basis': MRA_DIR,
            'status': 'missing_input',
            'reason': 'missing mra dir or input',
            'path': str(mra_dir),
        })
    else:
        done, reason, parse_error = run_complete_raman(mra_dir, load_calc)
        if done:
            status = 'complete'
        elif mra_dir.resolve() in running:
            status = 'running'
        elif parse_error:
            status = 'parse_error'
        else:
            status = 'pending'

        rows.append({
            'molecule': mol,
            'code': 'madness',
            'stage': 'raman',
            'basis': MRA_DIR,
            'status': status,
            'reason': reason,
            'path': str(mra_dir),
        })

    # DALTON stages
    basis_dirs = [d for d in mol_dir.iterdir() if d.is_dir() and d.name != MRA_DIR]
    basis_dirs = [
        d for d in basis_dirs
        if (d / OPTIMIZE_INPUT).exists() or (d / RAMAN_INPUT).exists() or any(d.glob('*.mol'))
    ]

    for basis_dir in sorted(basis_dirs, key=lambda p: p.name):
        basis = basis_dir.name
        optimize_input = basis_dir / OPTIMIZE_INPUT
        raman_input = basis_dir / RAMAN_INPUT
        base_mol = select_base_mol_file(mol, basis_dir)
        opt_mol = basis_dir / OPT_MOL_TEMPLATE.format(mol=mol, basis=basis)

        # Stage: DALTON optimize
        if base_mol is None or not optimize_input.exists():
            opt_status = 'missing_input'
            opt_reason = f'input={optimize_input.exists()}, base_mol={base_mol is not None}'
        elif opt_mol.exists():
            opt_status = 'complete'
            opt_reason = 'optimized molecule present'
        elif basis_dir.resolve() in running:
            opt_status = 'running'
            opt_reason = 'running in queue'
        elif any(basis_dir.glob('optimize*.out')):
            opt_status = 'pending_parse'
            opt_reason = 'optimize output exists, optimized mol not yet written'
        else:
            opt_status = 'pending'
            opt_reason = 'optimize not submitted or output missing'

        rows.append({
            'molecule': mol,
            'code': 'dalton',
            'stage': 'optimize',
            'basis': basis,
            'status': opt_status,
            'reason': opt_reason,
            'path': str(basis_dir),
        })

        # Stage: DALTON raman
        if not raman_input.exists():
            ram_status = 'missing_input'
            ram_reason = 'missing raman.dal'
        else:
            ram_done, ram_reason, ram_parse_error = run_complete_raman(basis_dir, load_calc)
            if ram_done:
                ram_status = 'complete'
            elif not opt_mol.exists():
                ram_status = 'waiting_opt'
                ram_reason = 'optimized molecule missing'
            elif basis_dir.resolve() in running:
                ram_status = 'running'
            elif ram_parse_error:
                ram_status = 'parse_error'
            else:
                ram_status = 'pending'

        rows.append({
            'molecule': mol,
            'code': 'dalton',
            'stage': 'raman',
            'basis': basis,
            'status': ram_status,
            'reason': ram_reason,
            'path': str(basis_dir),
        })

progress_df = pd.DataFrame(rows)
if not progress_df.empty:
    progress_df = progress_df.sort_values(['molecule', 'code', 'stage', 'basis']).reset_index(drop=True)

progress_df

In [ ]:
status_order = ['complete', 'running', 'pending', 'waiting_opt', 'pending_parse', 'parse_error', 'missing_input']
summary = (
    progress_df.assign(status=pd.Categorical(progress_df['status'], categories=status_order, ordered=True))
    .groupby(['code', 'stage', 'status'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=status_order, fill_value=0)
)
summary

In [ ]:
plot_colors = {
    'complete': '#2E8B57',
    'running': '#1f77b4',
    'pending': '#ff7f0e',
    'waiting_opt': '#bcbd22',
    'pending_parse': '#17becf',
    'parse_error': '#d62728',
    'missing_input': '#7f7f7f',
}

ax = summary.plot(
    kind='bar',
    stacked=True,
    figsize=(11, 5),
    color=[plot_colors[c] for c in summary.columns],
)
ax.set_title('Raman Run Status by Code/Stage')
ax.set_xlabel('Code, Stage')
ax.set_ylabel('Run Count')
ax.legend(title='Status', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

mol_completion = (
    progress_df.assign(done=progress_df['status'].eq('complete'))
    .groupby('molecule')['done']
    .mean()
    .mul(100.0)
    .reindex(sorted(progress_df['molecule'].unique(), key=molecule_sort_key))
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(mol_completion.index, mol_completion.values, color='#2ca02c')
ax.set_ylim(0, 100)
ax.set_ylabel('Completion (%)')
ax.set_xlabel('Molecule')
ax.set_title('Overall Completion by Molecule (all stages)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
todo_df = progress_df[~progress_df['status'].eq('complete')].copy()
todo_df = todo_df.sort_values(['molecule', 'code', 'stage', 'basis'])
todo_df[['molecule', 'code', 'stage', 'basis', 'status', 'reason']]

In [ ]:
# Optional: write a timestamped CSV snapshot
from datetime import datetime

snapshot_dir = RAMAN_ROOT / 'progress_snapshots'
snapshot_dir.mkdir(exist_ok=True)
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
snapshot_path = snapshot_dir / f'raman_progress_{stamp}.csv'
progress_df.to_csv(snapshot_path, index=False)
snapshot_path

In [ ]:
# Optional: trigger automated planning/submission from the notebook
SUBMIT = False
MAX_SUBMIT = 20
MOLECULES: list[str] = []
BASES: list[str] = []
SKIP_MADNESS = False
SKIP_DALTON = False
FORCE = False
GECKO_SRC_ARG = None
EXTRA_ARGS: list[str] = []

submit_script = RAMAN_ROOT / 'slurm_scripts' / 'submit_raman_jobs.py'
if not submit_script.exists():
    raise FileNotFoundError(f'Missing script: {submit_script}')

cmd = [sys.executable, str(submit_script)]
if SUBMIT:
    cmd.append('--submit')
if MAX_SUBMIT is not None:
    cmd += ['--max-submit', str(MAX_SUBMIT)]
if MOLECULES:
    cmd += ['--molecules', *MOLECULES]
if BASES:
    cmd += ['--bases', *BASES]
if SKIP_MADNESS:
    cmd.append('--skip-madness')
if SKIP_DALTON:
    cmd.append('--skip-dalton')
if FORCE:
    cmd.append('--force')
if GECKO_SRC_ARG:
    cmd += ['--gecko-src', str(Path(GECKO_SRC_ARG).expanduser())]
if EXTRA_ARGS:
    cmd += EXTRA_ARGS

print('Running command:')
print(' '.join(shlex.quote(x) for x in cmd))

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f'submit script failed with exit code {proc.returncode}')